# Answer or Abstain: Colab runs

Set the runtime to a GPU (Runtime > Change runtime type), then run the cells in order. Every inference cell is resumable: rerun it after a disconnect and it skips finished chunks.

## 1. GPU and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
REPO = "cs7650-proj-answer-or-abstain"
%cd /content
if not os.path.exists(REPO):
    !git clone -q https://github.com/DattaKansal/{REPO}.git
%cd {REPO}
!git pull -q
!pip install -q "transformers>=4.51" datasets accelerate pyarrow
!git log --oneline | head -3

## 2. Settings

With `USE_DRIVE = False` outputs stay on the Colab machine and are lost when the runtime is deleted. Set it to `True` to save to Google Drive (you will be asked to grant access).

In [ ]:
USE_DRIVE = False
MODEL = "Qwen/Qwen3-1.7B"
BATCH = 32  # lower to 16 or 8 on a T4 or if you hit out-of-memory

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cs7650_outputs"
else:
    OUT = "/content/outputs"
print(OUT)

## 3. Checks

In [ ]:
!python -m pytest -q

In [ ]:
# Where does the model put its probability at the answer position, with and without the "Answer:" prefill?
!python -m src.diagnose --model {MODEL} --n 200

## 4. Pilot (200 questions)

In [ ]:
!python -m src.infer --model {MODEL} --prompt neutral --save-hidden --limit 200 --batch-size {BATCH} --out {OUT}

## 5. Full runs

In [ ]:
# Neutral prompt: baseline answers, P3 logits and P4 hidden states
!python -m src.infer --model {MODEL} --prompt neutral --save-hidden --batch-size {BATCH} --out {OUT}

In [ ]:
# P1: the model decides, one pass per confidence target
for t in (0, 0.5, 0.75, 0.9):
    !python -m src.infer --model {MODEL} --prompt p1 --t {t} --batch-size {BATCH} --out {OUT}

In [ ]:
!du -sh {OUT}/*/*

## 6. Policies, table and figures

Fits the temperature (P3b) and the probe (P4), then scores every policy on the test split. Run `!git pull -q` first if the code changed since the runtime started.

In [ ]:
RUN_DIR = f"{OUT}/{MODEL.replace('/', '__')}"
!git pull -q
!python -m src.report --run-dir {RUN_DIR}

In [ ]:
from IPython.display import Image, display
for name in ("abstention_vs_t", "score_vs_t", "risk_coverage", "probe_by_layer"):
    display(Image(f"{RUN_DIR}/report/{name}.png", width=560))